# Exp11.0 — RSNN history internalization

Aggregation-only notebook for `d1_l1mem2_rsnn_fusion_internalization_v1`. It reads finalized artifacts only.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=None):
    p = Path.cwd() if start is None else Path(start)
    for candidate in (p, *p.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('repo root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_11_0_rsnn_history_internalization' / 'd1_l1mem2_rsnn_fusion_internalization_v1'
runs = pd.read_csv(root / 'run_metrics.csv')
methods = pd.read_csv(root / 'method_summary.csv')
contrasts = pd.read_csv(root / 'paired_contrast_summary.csv')
interactions = pd.read_csv(root / 'interaction_summary.csv')
gaps = pd.read_csv(root / 'temporal_gap_summary.csv')
activity = pd.read_csv(root / 'activity_summary.csv')
runs


## Native performance
Seeds are paired optimization replicates on one locked split.


In [ ]:
display_cols = ['l1_init','topology','seed','native_test_ba','window_test_ba','output_lif_test_ba','fusion_comm_valid_whole_ba','fusion_comm_valid_fixed250_ba','fusion_comm_valid_temporal_gap']
runs[display_cols].sort_values(['l1_init','topology','seed'])


## Temporal internalization
Primary mechanism: Fixed250-minus-whole should shrink from L1 to Fusion while Fusion whole/native BA rises.


In [ ]:
gap_view = gaps[['l1_init','topology','layer','support','fixed250_ba_mean','whole_ba_mean','temporal_gap_mean']]
gap_view.sort_values(['l1_init','topology','support','layer'])


In [ ]:
plot_data = gaps[gaps['support'] == 'valid'].copy()
for (l1_init, topology), group in plot_data.groupby(['l1_init','topology']):
    ordered = group.set_index('layer').loc[['l1','rsnn','fusion']].reset_index()
    plt.figure()
    plt.plot(ordered['layer'], ordered['temporal_gap_mean'], marker='o')
    plt.axhline(0, linewidth=1)
    plt.ylabel('Fixed250 BA - Whole BA')
    plt.title(f'{l1_init} / {topology}')
    plt.show()


## Paired contrasts


In [ ]:
contrasts


## Recurrent activity diagnostics


In [ ]:
runs[['l1_init','topology','seed','mean_abs_external_input','mean_abs_recurrent_input','recurrent_to_external_abs_ratio','recurrent_weight_norm']].sort_values(['l1_init','topology','seed'])
